In [ ]:
#Exercise 1 — Duplicate Detection & Removal

import pandas as pd

# Load dataset
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# Check for duplicates
rows_before = len(df)
print(f"Rows before: {rows_before}")
print(f"Duplicate rows found: {df.duplicated().sum()}")

# Remove duplicates
df = df.drop_duplicates()

# Verify
rows_after = len(df)
print(f"Rows after: {rows_after}")
print(f"Rows removed: {rows_before - rows_after}")

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
  

In [ ]:
#Exercise 2 — Handling Missing Values
import pandas as pd
from sklearn.impute import SimpleImputer

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# Identify missing values
print("Missing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Strategy 1: Drop 'Cabin' — too many missing (~77%)
df = df.drop(columns=['Cabin'])
print("\n-> Dropped 'Cabin' column")

# Strategy 2: Impute 'Age' with median using SimpleImputer
imputer = SimpleImputer(strategy='median')
df['Age'] = imputer.fit_transform(df[['Age']]).ravel()
print(f"-> Imputed 'Age' with median: {df['Age'].median()}")

# Strategy 3: Fill 'Embarked' with mode (most frequent value)
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
print(f"-> Filled 'Embarked' with mode: '{df['Embarked'].mode()[0]}'")

# Verify
print(f"\nRemaining missing values: {df.isnull().sum().sum()}")

In [3]:
# Exercise 3 — Feature Engineering

import pandas as pd
from sklearn.preprocessing import LabelEncoder

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
df = df.drop(columns=['Cabin'])
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Feature 1: Family Size
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
print(f"FamilySize range: {df['FamilySize'].min()} – {df['FamilySize'].max()}")

# Feature 2: IsAlone
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
print(f"Alone passengers: {df['IsAlone'].sum()}")

# Feature 3: Title extracted from Name
df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.', expand=False).str.strip()
print(f"\nAll titles found:\n{df['Title'].value_counts()}")

# Group rare titles
rare_titles = df['Title'].value_counts()[lambda x: x < 10].index
df['Title'] = df['Title'].replace(rare_titles, 'Rare')
print(f"\nTitles after grouping: {df['Title'].unique()}")

# Label encode Title
le = LabelEncoder()
df['Title_encoded'] = le.fit_transform(df['Title'])
print(f"\nLabel encoding mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

FamilySize range: 1 – 11
Alone passengers: 537

All titles found:
Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64

Titles after grouping: <StringArray>
['Mr', 'Mrs', 'Miss', 'Master', 'Rare']
Length: 5, dtype: str

Label encoding mapping: {'Master': np.int64(0), 'Miss': np.int64(1), 'Mr': np.int64(2), 'Mrs': np.int64(3), 'Rare': np.int64(4)}


In [ ]:
# EXERCISE 4


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.impute import SimpleImputer

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
df = df.drop(columns=['Cabin'])
df['Age'] = SimpleImputer(strategy='median').fit_transform(df[['Age']]).ravel()
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# ── Visualize before ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(y=df['Fare'], ax=axes[0], color='steelblue')
axes[0].set_title('Fare — Before')
sns.boxplot(y=df['Age'], ax=axes[1], color='mediumseagreen')
axes[1].set_title('Age — Before')
plt.tight_layout()
plt.show()

# ── Fare: IQR method ──
Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)
IQR = Q3 - Q1
fare_outliers = ((df['Fare'] < Q1 - 1.5*IQR) | (df['Fare'] > Q3 + 1.5*IQR)).sum()
print(f"Fare outliers (IQR): {fare_outliers}")

# Cap at 98th percentile
cap_98 = df['Fare'].quantile(0.98)
df['Fare_capped'] = df['Fare'].clip(upper=cap_98)
print(f"Fare capped at 98th percentile: {cap_98:.2f}")

# Log transformation
df['Fare_log'] = np.log1p(df['Fare'])
print("Fare log-transformed (log1p)")

# ── Age: Z-score method ──
z_scores = np.abs(stats.zscore(df['Age']))
age_outliers = (z_scores > 3).sum()
print(f"\nAge outliers (Z-score > 3): {age_outliers}")

# Cap at 98th percentile
age_cap = df['Age'].quantile(0.98)
df['Age_capped'] = df['Age'].clip(upper=age_cap)
print(f"Age capped at 98th percentile: {age_cap:.1f}")

# ── Visualize after ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(y=df['Fare_capped'], ax=axes[0], color='steelblue')
axes[0].set_title('Fare — After (Capped)')
sns.boxplot(y=df['Age_capped'], ax=axes[1], color='mediumseagreen')
axes[1].set_title('Age — After (Capped)')
plt.tight_layout()
plt.show()

1.9.0


In [4]:
# Exercise 5

import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
df = df.drop(columns=['Cabin'])
df['Age'] = SimpleImputer(strategy='median').fit_transform(df[['Age']]).ravel()
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['Age_capped'] = df['Age'].clip(upper=df['Age'].quantile(0.98))
df['Fare_log'] = np.log1p(df['Fare'].clip(upper=df['Fare'].quantile(0.98)))

# StandardScaler on Age (roughly normal after capping)
std_scaler = StandardScaler()
df['Age_scaled'] = std_scaler.fit_transform(df[['Age_capped']]).ravel()
print(f"Age_scaled — mean: {df['Age_scaled'].mean():.4f}, std: {df['Age_scaled'].std():.4f}")

# MinMaxScaler on Fare_log (bounded, still skewed)
mm_scaler = MinMaxScaler()
df['Fare_normalized'] = mm_scaler.fit_transform(df[['Fare_log']]).ravel()
print(f"Fare_normalized — min: {df['Fare_normalized'].min():.4f}, max: {df['Fare_normalized'].max():.4f}")

# MinMaxScaler on FamilySize
df['FamilySize_norm'] = mm_scaler.fit_transform(df[['FamilySize']]).ravel()
print(f"FamilySize_norm — min: {df['FamilySize_norm'].min():.4f}, max: {df['FamilySize_norm'].max():.4f}")

print(df[['Age_capped', 'Age_scaled', 'Fare_log', 'Fare_normalized', 'FamilySize', 'FamilySize_norm']].head())

Age_scaled — mean: 0.0000, std: 1.0006
Fare_normalized — min: 0.0000, max: 1.0000
FamilySize_norm — min: 0.0000, max: 1.0000
   Age_capped  Age_scaled  Fare_log  Fare_normalized  FamilySize  \
0        22.0   -0.570988  2.110213         0.393830           2   
1        38.0    0.687580  4.280593         0.798890           2   
2        26.0   -0.256346  2.188856         0.408508           1   
3        35.0    0.451599  3.990834         0.744812           2   
4        35.0    0.451599  2.202765         0.411103           1   

   FamilySize_norm  
0              0.1  
1              0.1  
2              0.0  
3              0.1  
4              0.0  


In [ ]:
# Exercise 6

import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
df = df.drop(columns=['Cabin'])
df['Age'] = SimpleImputer(strategy='median').fit_transform(df[['Age']]).ravel()
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.', expand=False).str.strip()
rare = df['Title'].value_counts()[lambda x: x < 10].index
df['Title'] = df['Title'].replace(rare, 'Rare')

# Identify categorical columns to encode
cat_cols = ['Sex', 'Embarked', 'Title']
print(f"Categorical columns to encode: {cat_cols}")

# One-Hot Encoding for nominal variables (Sex, Embarked, Title)
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=False)
new_cols = [c for c in df_encoded.columns if any(c.startswith(p + '_') for p in cat_cols)]
print(f"\nNew OHE columns created:\n{new_cols}")

# Example of Label Encoding (if an ordinal variable existed)
# Here shown on Pclass as a demonstration (ordinal: 1 > 2 > 3)
le = LabelEncoder()
df_encoded['Pclass_encoded'] = le.fit_transform(df_encoded['Pclass'])
print(f"\nPclass label encoded: {dict(zip(le.classes_, le.transform(le.classes_)))}")

print(f"\nDataset shape after encoding: {df_encoded.shape}")
print(df_encoded[new_cols].head())

In [5]:
#Exercise 7

import pandas as pd
from sklearn.impute import SimpleImputer

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
df = df.drop(columns=['Cabin'])
df['Age'] = SimpleImputer(strategy='median').fit_transform(df[['Age']]).ravel()
df['Age_capped'] = df['Age'].clip(upper=df['Age'].quantile(0.98))

# Step 1: Create age bins using pd.cut()
bins   = [0, 12, 18, 60, 100]
labels = ['Child', 'Teen', 'Adult', 'Senior']

df['AgeGroup'] = pd.cut(df['Age_capped'], bins=bins, labels=labels, right=True)
print("AgeGroup distribution:")
print(df['AgeGroup'].value_counts().sort_index())

# Step 2: One-Hot Encode AgeGroup using pd.get_dummies()
df = pd.get_dummies(df, columns=['AgeGroup'], prefix='AgeGroup')

age_group_cols = [c for c in df.columns if c.startswith('AgeGroup_')]
print(f"\nNew AgeGroup columns: {age_group_cols}")
print(df[age_group_cols].head(10))
print(f"\nFinal shape: {df.shape}")

AgeGroup distribution:
AgeGroup
Child      69
Teen       70
Adult     730
Senior     22
Name: count, dtype: int64

New AgeGroup columns: ['AgeGroup_Child', 'AgeGroup_Teen', 'AgeGroup_Adult', 'AgeGroup_Senior']
   AgeGroup_Child  AgeGroup_Teen  AgeGroup_Adult  AgeGroup_Senior
0           False          False            True            False
1           False          False            True            False
2           False          False            True            False
3           False          False            True            False
4           False          False            True            False
5           False          False            True            False
6           False          False            True            False
7            True          False           False            False
8           False          False            True            False
9           False           True           False            False

Final shape: (891, 16)
